# Silver Processing — Category

Cleans `ecommerce.bronze.bronze_category` into `ecommerce.silver.silver_category`:

* **All business columns** (`category_code`, `category_name`): trim spaces and strip trailing special characters
* **`category_code` in CAPS** — lowercase codes (`ce`, `app`, ...) become `CE`, `APP`, ..., matching `silver_brands`' canonical codes
* **Duplicate rows removed** — one row per category
* **Metadata columns** (`source_file`, `ingested_at`) pass through untouched

In [0]:
from pyspark.sql.functions import col, trim, upper
from pyspark.sql.functions import regexp_replace

bronze_table = "ecommerce.bronze.bronze_category"
target_table = "ecommerce.silver.silver_category"

df_bronze_category = spark.table(bronze_table)

# Clean a business column: trim spaces and strip any trailing special characters
def clean(column_name):
    return regexp_replace(trim(col(column_name)), "[^A-Za-z0-9]+$", "")

df_silver_category = (
    df_bronze_category
    .withColumn("category_code", upper(clean("category_code")))  # codes in CAPS
    .withColumn("category_name", clean("category_name"))
    .dropDuplicates(["category_code", "category_name"])  # remove duplicate rows
)

# Idempotent full load into the silver table
(
    df_silver_category.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
display(spark.table("ecommerce.silver.silver_category"))

In [0]:
# Check 1: one row per category_code, codes in CAPS
spark.sql("""
    SELECT category_code, COUNT(*) AS cnt
    FROM ecommerce.silver.silver_category
    GROUP BY category_code
    ORDER BY category_code
""").show()

# Check 2: no residual spaces, trailing special characters, or lowercase codes
spark.sql("""
    SELECT COUNT(*) AS dirty_rows
    FROM ecommerce.silver.silver_category
    WHERE category_code != upper(category_code)
       OR category_code != trim(category_code)
       OR category_code RLIKE '[^A-Za-z0-9]$'
       OR category_name != trim(category_name)
       OR category_name RLIKE '[^A-Za-z0-9]$'
""").show()